# Phase 5: Dashboard A4 (anymap-ts)

This notebook subscribes to queue, KPI, and optional congestion topics and visualizes live halftime status.

In [1]:
# Cell 2 purpose: Import dashboard, MQTT, and anymap-ts modules.
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path('../src').resolve()))

from anymap_ts import Map
from simulated_city import mqtt
from simulated_city.config import load_config
from simulated_city.dashboard_views import DashboardState, update_dashboard_state_from_topic
from simulated_city.topic_schema import topic_congestion_state, topic_kpi_metrics, topic_queue_state

In [5]:
# Cell 3 purpose: Load config, connect MQTT, and initialize map and state containers.
app_config = load_config()
state = DashboardState()

queue_topic = topic_queue_state()
kpi_topic = topic_kpi_metrics()
congestion_topic = topic_congestion_state()

mqtt_client = mqtt.connect_mqtt(app_config.mqtt, client_id_suffix='dashboard-a4')
print(f'Connected to MQTT broker at {app_config.mqtt.host}:{app_config.mqtt.port}')
print(f'Subscribing to: {queue_topic}')
print(f'Subscribing to: {kpi_topic}')
print(f'Subscribing to: {congestion_topic} (optional)')

dashboard_map = Map(center=(12.5683, 55.6761), zoom=17, height='560px')
dashboard_map.add_basemap('OpenStreetMap.Mapnik')
dashboard_map

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


Connected to MQTT broker at 127.0.0.1:1883
Subscribing to: stadium/a4/halftime/state/queues
Subscribing to: stadium/a4/halftime/metrics/kpi
Subscribing to: stadium/a4/halftime/state/congestion (optional)


In [ ]:
# Cell 4 purpose: Register MQTT callback to update dashboard state and map markers.
marker_ids = ['zone-a', 'zone-b', 'urinal-shared']

def _refresh_map_from_latest_queue():
    if not state.queue_trends:
        return
    latest = state.queue_trends[-1]

    for marker_id in marker_ids:
        try:
            dashboard_map.remove_marker(marker_id)
        except Exception:
            pass

    dashboard_map.add_marker(
        12.5678, 55.6762,
        name='zone-a',
        color='#1f77b4',
        popup=f'Zone A\nToilet: {latest.zone_a_toilet}\nCafe: {latest.zone_a_cafe}'
    )
    dashboard_map.add_marker(
        12.5690, 55.6760,
        name='zone-b',
        color='#ff7f0e',
        popup=f'Zone B\nToilet: {latest.zone_b_toilet}\nCafe: {latest.zone_b_cafe}'
    )
    dashboard_map.add_marker(
        12.5685, 55.6756,
        name='urinal-shared',
        color='#2ca02c',
        popup=f'Shared urinal queue: {latest.shared_mens_urinal}'
    )

def _on_message(client, userdata, msg):
    try:
        payload = json.loads(msg.payload.decode('utf-8'))
    except json.JSONDecodeError:
        return

    try:
        update_dashboard_state_from_topic(
            state,
            msg.topic,
            payload,
            topic_queue_state=queue_topic,
            topic_kpi_metrics=kpi_topic,
            topic_congestion_state=congestion_topic,
        )
    except ValueError:
        return

    if msg.topic == queue_topic:
        _refresh_map_from_latest_queue()

mqtt_client.on_message = _on_message
print('Dashboard callback registered.')

Dashboard callback registered.


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=

In [7]:
# Cell 5 purpose: Subscribe to topics and process incoming messages for a timed window.
mqtt_client.subscribe(queue_topic, qos=1)
mqtt_client.subscribe(kpi_topic, qos=1)
mqtt_client.subscribe(congestion_topic, qos=1)
print('Subscriptions active. Listening for 30 seconds...')

start = time.time()
while time.time() - start < 30:
    elapsed = int(time.time() - start)
    if elapsed % 5 == 0:
        latest_ts = state.queue_trends[-1].timestamp_s if state.queue_trends else None
        print(f'  t+{elapsed:02d}s | queue_points={len(state.queue_trends)} latest_queue_ts={latest_ts}')
    time.sleep(1)

print('Dashboard listening window finished.')

Subscriptions active. Listening for 30 seconds...
  t+00s | queue_points=0 latest_queue_ts=None


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


  t+05s | queue_points=0 latest_queue_ts=None


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


  t+10s | queue_points=0 latest_queue_ts=None


...

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


  t+25s | queue_points=0 latest_queue_ts=None


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


Dashboard listening window finished.


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=

In [8]:
# Cell 6 purpose: Print summary metrics and disconnect the dashboard client.
print('\n=== Dashboard Summary ===')
print(f'Queue trend points: {len(state.queue_trends)}')
if state.latest_kpi is not None:
    print(f'Latest KPI timestamp: {state.latest_kpi.timestamp_s}')
    print(f'Average wait (s): {state.latest_kpi.average_wait_s:.2f}')
    print(f'Missed kickoff count: {state.latest_kpi.missed_kickoff_count}')
    print(f'Made kickoff count: {state.latest_kpi.made_kickoff_count}')
    print(f'Stayed seated count: {state.latest_kpi.stayed_seated_count}')
    print(f'Went down count: {state.latest_kpi.went_down_count}')
    print(f'Went down and made back count: {state.latest_kpi.went_down_made_back_count}')
    print(f'Wait P50/P95/P99: {state.latest_kpi.wait_percentiles_s["P50"]:.2f} / {state.latest_kpi.wait_percentiles_s["P95"]:.2f} / {state.latest_kpi.wait_percentiles_s["P99"]:.2f}')
else:
    print('No KPI payload received on stadium/a4/halftime/metrics/kpi yet.')

if state.latest_congestion is not None:
    print(f'Congestion | zone_a_blocked={state.latest_congestion.zone_a_blocked} zone_b_blocked={state.latest_congestion.zone_b_blocked}')
else:
    print('No congestion payload received (optional topic).')

connector = getattr(mqtt_client, '_simcity_connector', None)
if connector is not None:
    connector.disconnect()
    print('Disconnected from MQTT broker.')


=== Dashboard Summary ===
Queue trend points: 0
No KPI payload received on stadium/a4/halftime/metrics/kpi yet.
No congestion payload received (optional topic).


Disconnected from MQTT broker (reason=Normal disconnection). Reconnecting...


Disconnected from MQTT broker.
